In [1]:
from typing import TypedDict , List , Dict
from langgraph.graph import StateGraph , START , END
import random

In [2]:
class AgentState(TypedDict):
    player_name: str
    guesses: List[int]
    attempts: int
    lower_bound: int
    upper_bound: int
    target: int
    result: str

In [3]:
def guess_node(state: AgentState) -> AgentState:
    """Guesses the midpoint between bounds"""
    guess = (state['lower_bound'] + state['upper_bound']) // 2
    state['guesses'].append(guess)
    state['attempts'] += 1
    print(f"Attempt {state['attempts']}: Guessing {guess}")
    return state

def hint_node(state: AgentState) -> AgentState:
    """Adjusts bounds based on last guess vs target"""
    last_guess = state['guesses'][-1]
    if last_guess < state['target']:
        print(f"  → Higher! (guess was {last_guess})")
        state['lower_bound'] = last_guess + 1
    elif last_guess > state['target']:
        print(f"  → Lower! (guess was {last_guess})")
        state['upper_bound'] = last_guess - 1
    return state

# --- Conditional edge ---

def should_continue(state: AgentState) -> str:
    last_guess = state['guesses'][-1]
    if last_guess == state['target']:
        state['result'] = f"🎉 {state['player_name']} found it! Number was {state['target']} in {state['attempts']} attempts!"
        return 'found'
    elif state['attempts'] >= 7:
        state['result'] = f"❌ Max attempts reached! Number was {state['target']}, last guess was {last_guess}"
        return 'max_reached'
    else:
        return 'keep_guessing'

In [4]:
graph = StateGraph(AgentState)

graph.add_node("guess_node", guess_node)
graph.add_node("hint_node", hint_node)

graph.add_edge(START, "guess_node")
graph.add_edge("guess_node", "hint_node")
graph.add_conditional_edges("hint_node", should_continue, {
    "found": END,
    "max_reached": END,
    "keep_guessing": "guess_node"   # 👈 loops back
})

app = graph.compile()

In [5]:
target = random.randint(1, 20)
print(f"(Target is: {target})\n")

result = app.invoke({
    "player_name": "Student",
    "guesses": [],
    "attempts": 0,
    "lower_bound": 1,
    "upper_bound": 20,
    "target": target,
    "result": ""
})


(Target is: 19)

Attempt 1: Guessing 10
  → Higher! (guess was 10)
Attempt 2: Guessing 15
  → Higher! (guess was 15)
Attempt 3: Guessing 18
  → Higher! (guess was 18)
Attempt 4: Guessing 19


In [7]:
print("\n" + result['result'])


In [8]:
print(f"Guesses were: {result['guesses']}")

Guesses were: [10, 15, 18, 19]
